# SPAD photon-stream search on T4 x2: police + randomized-conditions batch

Attach the private `spad-stream-design` dataset, choose **GPU T4 x2**, then use **Save Version > Save & Run All**.
A saved version keeps running after the browser closes. Download `spad_runs_police.zip` from the output when it finishes.

To continue an unfinished sweep, attach the previous output as an input and set `PREVIOUS` below.

This batch: every job trains with a random photon count (16-256) and extra ambient (0-0.6) per sample. Six jobs use the label-free physics police; two (`*-nopolice`) are the ablation. Upload `spad-stream-design.zip` from this folder as a **new version** of the dataset first: this notebook needs `police.py`, which older bundles lack.

In [ ]:
import glob, os, shutil, subprocess, sys

matches = glob.glob('/kaggle/input/**/code/spad/training/train_stream.py', recursive=True)
assert matches, 'Attach the spad-stream-design dataset'
BUNDLE = matches[0].rsplit('/code/', 1)[0]
WORK = '/kaggle/working'
CODE, DATA, RUNS = f'{WORK}/code', f'{BUNDLE}/data', f'{WORK}/runs'
SESSION_SECONDS = 8 * 3600   # stop safely below Kaggle's session limit
PREVIOUS = None              # for example '/kaggle/input/previous-output/runs'

shutil.copytree(f'{BUNDLE}/code', CODE, dirs_exist_ok=True)
if PREVIOUS:
    shutil.copytree(PREVIOUS, RUNS, dirs_exist_ok=True)
import json
splits = {k: v['split'] for k, v in json.load(open(f'{DATA}/processed/metadata.json'))['captures'].items()}
held_out = [k for k, v in splits.items() if v not in ('train', 'architecture', 'validation')]
held_out += [p for p in os.listdir(f'{DATA}/processed') if p.split('.')[0].removeprefix('labels_') in {'7', '8', '9', '10'} or p.startswith(('parking', 'labels_parking'))]
assert not held_out, f'Held-out captures must not be here: {held_out}'
print(BUNDLE)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose GPU T4 x2 under Accelerator'
GPUS = torch.cuda.device_count()
print(torch.__version__, GPUS, [torch.cuda.get_device_name(i) for i in range(GPUS)])

CPU and CUDA must agree exactly. Stop here if this fails.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_device', '-v'], cwd=CODE, check=True)

Check whether larger batches help before spending quota. Set `BATCH` and `POPULATION` below from these numbers.

In [ ]:
bench = subprocess.run([sys.executable, '-m', 'spad.evaluation.bench_stream', '--device', 'cuda'], cwd=CODE, check=True, capture_output=True, text=True)
print(bench.stdout)
os.makedirs(RUNS, exist_ok=True)
open(f'{RUNS}/benchmark_t4.json', 'w').write(bench.stdout)

In [ ]:
# Photon streams are now up to 256 long (4x the old 64), so each pass costs ~4x.
BATCH, POPULATION, SEARCH_RAYS, GENERATIONS = 1024, 128, 1024, 600
subprocess.run([sys.executable, 'kaggle/launch.py', '--jobs', 'kaggle/jobs.txt', '--out', RUNS, '--data', DATA,
                '--gpus', str(GPUS), '--max-seconds', str(SESSION_SECONDS),
                '--batch-size', str(BATCH), '--population', str(POPULATION),
                '--search-rays', str(SEARCH_RAYS), '--generations', str(GENERATIONS)],
               cwd=CODE, check=True)

In [ ]:
import json
rows = []
for path in glob.glob(f'{RUNS}/*/summary.json'):
    s = json.load(open(path))
    rows.append((s['validation_fresh_photons']['mae_m'], os.path.basename(os.path.dirname(path)), s['seconds']))
for mae, name, seconds in sorted(rows):
    print(f'{name:22s} MAE {mae:.3f} m  {seconds/60:.1f} min')
shutil.make_archive(f'{WORK}/spad_runs_police', 'zip', RUNS)